In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("202425_all_state_funded_pupils_characteristics_and_geography_breakdowns_revised.csv")
df2 = pd.read_csv('File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv')
pd.set_option('display.max_rows', None)

C:\Users\Olesya Drozhzhina\AppData\Local\Temp\ipykernel_19696\1071244348.py:3: DtypeWarning: Columns (0: new_la_code, 1: la_name, 2: school_count, 3: attainment8_sum, 4: attainment8_average, 5: engmath_entering_total, 6: engmath_entering_percent, 7: engmath_95_total, 8: engmath_95_percent, 9: engmath_94_total, 10: engmath_94_percent, 11: ebacc_entering_total, 12: ebacc_95_total, 13: ebacc_95_percent, 14: ebacc_94_total, 15: ebacc_94_percent, 16: ebacc_aps_sum, 17: ebacc_aps_average, 18: progress8_pupil_count, 19: gcse_entering_total, 20: gcse_entering_percent, 21: gcse_91_total, 22: gcse_91_percent, 23: ebacceng_entering_total, 24: ebacceng_entering_percent, 25: ebaccmat_entering_total, 26: ebaccmat_entering_percent, 27: ebaccsci_entering_total, 28: ebacchum_entering_total, 29: ebacclan_entering_total, 30: ebacceng_95_total, 31: ebacceng_95_percent, 32: ebaccmat_95_total, 33: ebaccmat_95_percent, 34: ebaccsci_95_total, 35: ebaccsci_95_percent, 36: ebacceng_94_total, 37: ebacceng_94_per

In [9]:
#df.shape
#df.head()
#df.columns.tolist()
# df[df["geographic_level"] == "Regional"]['region_name'].value_counts()
# df[df["geographic_level"] == "Local authority"]['la_name'].value_counts()
df_loc=df[df["geographic_level"] == "Local authority"].copy()

#df_loc["breakdown_topic"].value_counts()
#df_loc[['la_name','new_la_code']].drop_duplicates().reset_index(drop = True)



In [ ]:
# dataset for q1
cols_total = [
    'time_period',
    'new_la_code',
    'la_name',
    'school_count',
    'pupil_count',
    'attainment8_average',
    'progress8_average',
    'progress8_pupil_percent',
    'ebacc_aps_average',
    'ebacc_entering_percent',
]
df_total = df_loc[(df_loc["breakdown_topic"] == "Total")].copy()
df_total = df_total[cols_total]

# EDA
df_total.head()
df_total.groupby(['time_period', 'la_name']).size().value_counts() # To check that after filtering to the total breakpoint we have only one row for each pair la+year
df_total.isna().sum().sort_values(ascending=False) # no Null values
z_counts = (df_total == 'z').sum()
z_counts[z_counts > 0].sort_values(ascending=False) # where and how many z we have
df_total[df_total["progress8_average"] == "z"]["time_period"].value_counts()

df_total = df_total.replace('z', np.nan) #replace z wihr nulls to be able to change datatype to numeric
# checking datatypes, for some reason school count - 'object' should be int
# after checking what dtypes are in this column, found that there are 8 'str', 
# after checking these 8 'str' - they look like numbers so safely could be changed to int
# df_total.dtypes
# df_total['school_count'].map(type).value_counts() // what dtypes are in this column
# df_total[df_total['school_count'].map(type) == str][['la_name', 'time_period', 'school_count']] //checking why 'str'

numeric_cols = [
    'school_count',
    'pupil_count',
    'attainment8_average',
    'progress8_average',
    'progress8_pupil_percent',
    'ebacc_aps_average',
    'ebacc_entering_percent'
]

for col in numeric_cols:
    df_total[col] = pd.to_numeric(df_total[col])

df_total['time_period'] = df_total['time_period'].astype(str)

df_total.dtypes

In [ ]:
# dataset for q3:
cols_subjects = [
    'time_period',
    'new_la_code',
    'la_name',
    'ebacceng_aps_average',
    'ebaccmat_aps_average',
    'ebaccsci_aps_average',
    'ebacchum_aps_average',
    'ebacclan_aps_average',
    'valueaddedsci_average',
    'valueaddedhum_average',
    'valueaddedlan_average'
]
df_subjects = df_loc[(df_loc["breakdown_topic"] == "Total")].copy()
df_subjects = df_subjects[cols_subjects]

df_subjects.dtypes
# need to change datatypes the same way like for q1

In [ ]:
# q4&q5
df_gender = df[df["breakdown_topic"] == "Sex"].copy()
df_fsm = df[df["breakdown_topic"] == "FSM status"].copy()
df_disadv = df[df["breakdown_topic"] == "Disadvantage status"].copy()
df_lang = df[df["breakdown_topic"] == "First language"].copy()
#df_prior = df[df["breakdown_topic"] == "KS2 scaled score group"].copy() // do we need this??
#df_ethnicity = df[df["breakdown_topic"] == "Ethnicity"].copy() // could be difficult, a lot of splitting
# columns for these datasets??

In [ ]:
# what LA we have to map with imd dataset
authorities_df1 = df_loc[["new_la_code", "la_name"]].drop_duplicates().sort_values("la_name").reset_index(drop=True)

authorities_df1

In [ ]:
df2.head()
df2.columns.tolist()

In [ ]:

authorities_df2 = df2[['Local Authority District code (2019)', 'Local Authority District name (2019)']].drop_duplicates().sort_values("Local Authority District name (2019)").reset_index(drop=True)

authorities_df2

In [60]:
dfe_codes = set(df_loc["new_la_code"].dropna().astype(str).str.strip())
imd_codes = set(df2["Local Authority District code (2019)"].dropna().astype(str).str.strip())

matching_codes = dfe_codes & imd_codes
only_in_dfe = dfe_codes - imd_codes
only_in_imd = imd_codes - dfe_codes

print("DFE codes:", len(dfe_codes))
print("IMD codes:", len(imd_codes))
print("Matching:", len(matching_codes))
print("Only in DFE:", len(only_in_dfe))
print("Only in IMD:", len(only_in_imd))

only_dfe_df = df_loc[
    df_loc["new_la_code"].astype(str).str.strip().isin(only_in_dfe)
][["new_la_code", "la_name"]].drop_duplicates().sort_values("la_name")

only_dfe_df.to_csv("only_in_dfe.csv", index=False)

only_imd_df = df2[
    df2["Local Authority District code (2019)"].astype(str).str.strip().isin(only_in_imd)
][["Local Authority District code (2019)", "Local Authority District name (2019)"]]\
.drop_duplicates()\
.sort_values("Local Authority District name (2019)")

only_imd_df.to_csv("only_in_imd.csv", index=False)

matching_df = df[
    df["new_la_code"].astype(str).str.strip().isin(matching_codes)
][["new_la_code", "la_name"]]\
.drop_duplicates()\
.sort_values("la_name")

matching_df.to_csv("matching_codes.csv", index=False)



DFE codes: 157
IMD codes: 317
Matching: 124
Only in DFE: 33
Only in IMD: 193


In [62]:
# look up table which match la codes with la/district codes
df_lookup = pd.read_csv('Look_up.csv')
df_lookup.columns.to_list()
df_lookup.head()

,LAD20CD,LAD20NM,CTY20CD,CTY20NM,FID
0,E07000138,Lincoln,E10000019,Lincolnshire,1
1,E07000203,Mid Suffolk,E10000029,Suffolk,2
2,E08000026,Coventry,E11000005,West Midlands,3
3,E07000086,Eastleigh,E10000014,Hampshire,4
4,E07000008,Cambridge,E10000003,Cambridgeshire,5


In [ ]:
#Merge IMD district data with lookup
imd_lookup = df2.merge(
    df_lookup,
    left_on="Local Authority District code (2019)",
    right_on="LAD20CD",
    how="left"
)
imd_lookup.head()

In [64]:
# aggregate imd to LA level
imd_la = (
    imd_lookup
    .groupby(["CTY20CD", "CTY20NM"], as_index=False)
    .agg({
        "Index of Multiple Deprivation (IMD) Score": "mean",
        "Income Score (rate)": "mean",
        "Employment Score (rate)": "mean",
        "Health Deprivation and Disability Score": "mean",
        "Crime Score": "mean",
        "Barriers to Housing and Services Score": "mean",
        "Living Environment Score": "mean"
    })
)

In [65]:
# now we can merge
final_q1 = df_total.merge(
    imd_la,
    left_on="new_la_code",
    right_on="CTY20CD",
    how="left"
)

In [ ]:
final_q1[final_q1["Index of Multiple Deprivation (IMD) Score"].isna()][
    ["new_la_code", "la_name"]
].drop_duplicates()

In [ ]:
imd_lookup["CTY20CD"].isna().sum()
imd_lookup[imd_lookup["CTY20CD"].isna()][
    "Local Authority District code (2019)"
].nunique()
imd_lookup[imd_lookup["CTY20CD"].isna()][
    ["Local Authority District code (2019)",
     "Local Authority District name (2019)"]
].drop_duplicates()

,Local Authority District code (2019),Local Authority District name (2019)
11596,E06000001,Hartlepool
11649,E06000002,Middlesbrough
11730,E06000003,Redcar and Cleveland
11809,E06000004,Stockton-on-Tees
11919,E06000005,Darlington
11978,E06000006,Halton
12057,E06000007,Warrington
12180,E06000008,Blackburn with Darwen
12269,E06000009,Blackpool
12363,E06000010,"Kingston upon Hull, City of"
